# Recommendations with IBM

## Project: IBM Watson Studio – Article Recommendation System

In this notebook, we analyze user-article interactions on the IBM Watson Studio platform and build several recommendation engines:

1. **Exploratory Data Analysis** – Understand the data
2. **Rank-Based Recommendations** – Recommend the most popular articles
3. **User-User Collaborative Filtering** – Recommend based on similar users
4. **Content-Based Recommendations** – Recommend using NLP & clustering
5. **Matrix Factorization (SVD)** – Recommend using latent features


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib
import pickle

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.cluster import KMeans
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.decomposition import TruncatedSVD

import warnings
warnings.filterwarnings('ignore')

%matplotlib inline

# Load data
df = pd.read_csv('data/user-item-interactions.csv')
df_content = pd.read_csv('data/articles_community.csv')

del df['Unnamed: 0']
del df_content['Unnamed: 0']

df.head()

In [ ]:
df_content.head()

---
## Part I: Exploratory Data Analysis

We'll explore the dataset to understand its structure and key statistics.


In [ ]:
# Shape and basic info
print('Shape of interactions df:', df.shape)
print('Shape of content df:', df_content.shape)
print()
print('Columns in interactions df:', df.columns.tolist())
print('Columns in content df:', df_content.columns.tolist())

In [ ]:
# 1. What is the number of unique articles that have an interaction with a user?
unique_articles = df['article_id'].nunique()
print(f'Unique articles interacted with: {unique_articles}')

# 2. What is the number of unique users?
unique_users = df['email'].nunique()
print(f'Unique users: {unique_users}')

# 3. Total number of user-article interactions
user_article_interactions = df.shape[0]
print(f'Total user-article interactions: {user_article_interactions}')

# 4. Most viewed article
most_viewed_article_id = str(df['article_id'].value_counts().index[0])
max_views = df['article_id'].value_counts().iloc[0]
print(f'Most viewed article ID: {most_viewed_article_id}')
print(f'Max views by single article: {max_views}')

# 5. Max user views
max_views_by_user = df.groupby('email')['article_id'].count().max()
print(f'Max views by a single user: {max_views_by_user}')

# 6. Median interactions per user
median_val = df.groupby('email')['article_id'].count().median()
print(f'Median user interactions: {median_val}')

# 7. Total articles in the content df
total_articles = df_content['article_id'].nunique()
print(f'Total articles available: {total_articles}')

In [ ]:
# Solution dictionary for Part I tests
sol_1_dict = {
    '`50% of users interact with _____ number of articles or fewer.`': median_val,
    '`The total number of user-article interactions in the dataset is ______.`': user_article_interactions,
    '`The maximum number of articles that the same user has interacted with is ______.`': max_views_by_user,
    '`The maximum number of times an article has been viewed is ______.`': max_views,
    '`The most viewed article in the dataset as a string is ______.`': most_viewed_article_id,
    '`The number of unique articles that have at least one interaction are ______.`': unique_articles,
    '`The number of unique users in the dataset is ______`': unique_users,
    '`The number of unique articles on the IBM platform`': total_articles
}

# Display summary
for k, v in sol_1_dict.items():
    print(f'{k}: {v}')

In [ ]:
# Visualize distribution of user interactions
user_interaction_counts = df.groupby('email')['article_id'].count()

plt.figure(figsize=(10, 5))
plt.hist(user_interaction_counts, bins=50, color='steelblue', edgecolor='white')
plt.title('Distribution of User Interactions', fontsize=14)
plt.xlabel('Number of Articles Interacted With')
plt.ylabel('Number of Users')
plt.yscale('log')
plt.tight_layout()
plt.show()

print(f'\nUser interaction stats:')
print(user_interaction_counts.describe())

---
## Part II: Rank-Based Recommendations

We find the most popular articles based on interaction count and recommend them — especially useful for new users.


In [ ]:
def get_top_articles(n, df):
    """
    Return the top n article titles based on interaction count.
    
    INPUT:
        n   - (int) number of top articles to return
        df  - (pd.DataFrame) the interactions dataframe
    OUTPUT:
        top_articles - (list) a list of the top 'n' article titles 
    """
    top_articles = (df.groupby('title')['article_id']
                      .count()
                      .sort_values(ascending=False)
                      .head(n)
                      .index
                      .tolist())
    return top_articles


def get_top_article_ids(n, df):
    """
    Return the top n article ids based on interaction count.
    
    INPUT:
        n   - (int) number of top articles to return
        df  - (pd.DataFrame) the interactions dataframe
    OUTPUT:
        top_articles - (list) a list of the top 'n' article ids as strings
    """
    top_articles = (df.groupby('article_id')['title']
                      .count()
                      .sort_values(ascending=False)
                      .head(n)
                      .index
                      .astype(str)
                      .tolist())
    return top_articles


print('Top 10 Articles:')
print(get_top_articles(10, df))
print()
print('Top 10 Article IDs:')
print(get_top_article_ids(10, df))

---
## Part III: User-User Collaborative Filtering

We build a user-item matrix and find similar users to make recommendations.


In [ ]:
def create_user_item_matrix(df):
    """
    Return a matrix with user ids on the rows and article ids on the columns.
    Each cell is 1 if the user has interacted with the article, 0 otherwise.
    
    INPUT:
        df - pandas dataframe with article_id, title, email columns
    OUTPUT:
        user_item - user item matrix (pd.DataFrame)
    """
    user_item = (df.groupby(['email', 'article_id'])['title']
                   .count()
                   .unstack()
                   .fillna(0)
                   .applymap(lambda x: 1 if x > 0 else 0))
    return user_item


user_item = create_user_item_matrix(df)
print('User-Item Matrix Shape:', user_item.shape)
user_item.head()

In [ ]:
def find_similar_users(user_id, user_item):
    """
    Find similar users based on shared article interactions (dot product).
    Users are ordered by similarity (most similar first),
    then by total interactions (most active first) for ties.
    
    INPUT:
        user_id   - (int) the user_id of the individual you want to find similar users for
        user_item - (pd.DataFrame) matrix of users by articles
    OUTPUT:
        similar_users - (list) ordered list of similar users, excluding user_id itself
    """
    # Compute dot product similarity
    user_vec = user_item.loc[user_id].values.reshape(1, -1)
    similarity = user_item.values.dot(user_vec.T).flatten()
    
    sim_series = pd.Series(similarity, index=user_item.index)
    
    # Sort by similarity desc, then by interaction count desc (for ties)
    interaction_counts = user_item.sum(axis=1)
    sim_df = pd.DataFrame({'similarity': sim_series, 'interactions': interaction_counts})
    sim_df = sim_df.sort_values(['similarity', 'interactions'], ascending=[False, False])
    
    # Remove the user themselves
    similar_users = sim_df.index[sim_df.index != user_id].tolist()
    return similar_users


# Test
test_user = user_item.index[0]
similar = find_similar_users(test_user, user_item)
print(f'Top 5 similar users to user "{test_user}":')
print(similar[:5])

In [ ]:
def get_article_names(article_ids, df):
    """
    Get article names from article IDs.
    
    INPUT:
        article_ids - (list) list of article ids
        df          - (pd.DataFrame) interactions dataframe
    OUTPUT:
        article_names - (list) list of article names
    """
    article_names = []
    for art_id in article_ids:
        title = df[df['article_id'] == float(art_id)]['title'].values
        if len(title) > 0:
            article_names.append(title[0])
    return article_names


def get_user_articles(user_id, user_item):
    """
    Return article ids and names for articles the user has interacted with.
    
    INPUT:
        user_id   - (int) the user id
        user_item - (pd.DataFrame) user-item matrix
    OUTPUT:
        article_ids   - (list) articles the user has interacted with
        article_names - (list) corresponding article names
    """
    article_ids = user_item.loc[user_id]
    article_ids = article_ids[article_ids == 1].index.astype(str).tolist()
    article_names = get_article_names(article_ids, df)
    return article_ids, article_names


# Test
art_ids, art_names = get_user_articles(test_user, user_item)
print(f'Articles interacted with by "{test_user}":')
print(art_names[:5])

In [ ]:
def user_user_recs(user_id, m=10):
    """
    Return top m article recommendations for a user based on user-user collaborative filtering.
    
    INPUT:
        user_id - (int) a user id
        m       - (int) the number of recommendations to provide
    OUTPUT:
        recs - (list) a list of recommendations for the user
    """
    # Articles already seen by user
    seen_ids, _ = get_user_articles(user_id, user_item)
    seen_ids = set(seen_ids)
    
    # Find similar users
    similar_users = find_similar_users(user_id, user_item)
    
    recs = []
    for sim_user in similar_users:
        sim_user_arts, _ = get_user_articles(sim_user, user_item)
        new_arts = [a for a in sim_user_arts if a not in seen_ids and a not in recs]
        recs.extend(new_arts)
        if len(recs) >= m:
            break
    
    return recs[:m]


# Test
recs = user_user_recs(test_user, 10)
print(f'Top 10 recommendations for "{test_user}":')
print(get_article_names(recs, df))

In [ ]:
def user_user_recs_part2(user_id, m=10):
    """
    Improved collaborative filtering:
    - Ranked by most similar users first
    - Ties broken by number of user interactions
    - Articles ranked by most interactions
    
    INPUT:
        user_id - (int) a user id
        m       - (int) the number of recommendations
    OUTPUT:
        recs        - (list) recommended article ids
        rec_names   - (list) corresponding article names
    """
    seen_ids, _ = get_user_articles(user_id, user_item)
    seen_ids = set(seen_ids)
    
    # Get article popularity for ranking
    article_popularity = df.groupby('article_id')['title'].count().to_dict()
    
    similar_users = find_similar_users(user_id, user_item)
    
    candidate_articles = {}
    for sim_user in similar_users:
        sim_arts, _ = get_user_articles(sim_user, user_item)
        for art in sim_arts:
            if art not in seen_ids and art not in candidate_articles:
                candidate_articles[art] = article_popularity.get(float(art), 0)
    
    # Sort by popularity
    recs = sorted(candidate_articles, key=candidate_articles.get, reverse=True)[:m]
    rec_names = get_article_names(recs, df)
    return recs, rec_names


recs2, rec_names2 = user_user_recs_part2(test_user, 10)
print(f'Improved recommendations for "{test_user}":')
for i, name in enumerate(rec_names2, 1):
    print(f'  {i}. {name}')

In [ ]:
def make_recs_for_new_user(n=10):
    """
    Provide recommendations for new users using rank-based approach.
    New users have no interaction history, so we fall back to popularity.
    
    INPUT:
        n - (int) number of recommendations
    OUTPUT:
        top_articles - (list) top n article names
    """
    top_articles = get_top_articles(n, df)
    print(f'Recommendations for new user (rank-based, top {n}):')
    for i, art in enumerate(top_articles, 1):
        print(f'  {i}. {art}')
    return top_articles


new_user_recs = make_recs_for_new_user(10)

---
## Part IV: Content-Based Recommendations

Using TF-IDF on article text and KMeans clustering to group similar articles.


In [ ]:
# Prepare article text by combining title and description
df_content['doc_full'] = (df_content['doc_full_name'].fillna('') + ' ' +
                          df_content['doc_description'].fillna(''))

# TF-IDF vectorization
tfidf = TfidfVectorizer(stop_words='english', max_features=5000)
tfidf_matrix = tfidf.fit_transform(df_content['doc_full'])

print('TF-IDF matrix shape:', tfidf_matrix.shape)

In [ ]:
# Find optimal number of clusters using the Elbow Method
inertias = []
K_range = range(2, 15)

for k in K_range:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    km.fit(tfidf_matrix)
    inertias.append(km.inertia_)

plt.figure(figsize=(10, 5))
plt.plot(K_range, inertias, 'bo-', markersize=8)
plt.title('KMeans Elbow Method – Optimal Cluster Count', fontsize=14)
plt.xlabel('Number of Clusters (k)')
plt.ylabel('Inertia')
plt.xticks(K_range)
plt.grid(True, linestyle='--', alpha=0.5)
plt.tight_layout()
plt.show()

In [ ]:
# Fit KMeans with optimal k (choose based on elbow)
optimal_k = 6
kmeans = KMeans(n_clusters=optimal_k, random_state=42, n_init=10)
df_content['cluster'] = kmeans.fit_predict(tfidf_matrix)

# Show cluster distribution
print('Cluster distribution:')
print(df_content['cluster'].value_counts().sort_index())

plt.figure(figsize=(8, 4))
df_content['cluster'].value_counts().sort_index().plot(kind='bar', color='steelblue', edgecolor='white')
plt.title(f'Articles per Cluster (k={optimal_k})', fontsize=13)
plt.xlabel('Cluster')
plt.ylabel('Number of Articles')
plt.tight_layout()
plt.show()

In [ ]:
def make_content_recs(article_id, n=10):
    """
    Recommend articles similar to a given article using cosine similarity on TF-IDF.
    
    INPUT:
        article_id - (float/int) the source article id
        n          - (int) number of recommendations
    OUTPUT:
        rec_ids   - (list) recommended article ids
        rec_names - (list) corresponding article names
    """
    # Find the index of the article in df_content
    matches = df_content[df_content['article_id'] == article_id]
    if matches.empty:
        print(f'Article {article_id} not found in content dataframe.')
        return [], []
    
    idx = matches.index[0]
    article_vec = tfidf_matrix[idx]
    
    # Cosine similarity with all other articles
    sim_scores = cosine_similarity(article_vec, tfidf_matrix).flatten()
    sim_series = pd.Series(sim_scores, index=df_content.index)
    sim_series = sim_series.drop(idx).sort_values(ascending=False)
    
    top_indices = sim_series.head(n).index
    rec_ids = df_content.loc[top_indices, 'article_id'].astype(str).tolist()
    rec_names = df_content.loc[top_indices, 'doc_full_name'].tolist()
    
    return rec_ids, rec_names


# Test content-based recommendations
sample_article_id = df_content['article_id'].iloc[0]
sample_title = df_content['doc_full_name'].iloc[0]
print(f'Content-based recommendations for: "{sample_title}"\n')

c_rec_ids, c_rec_names = make_content_recs(sample_article_id, 10)
for i, name in enumerate(c_rec_names, 1):
    print(f'  {i}. {name}')

---
## Part V: Matrix Factorization (SVD)

We use Singular Value Decomposition to build a latent-factor model and evaluate its performance.


In [ ]:
# Create numpy user-item matrix
user_item_matrix = user_item.values.astype(float)
print('User-Item Matrix shape:', user_item_matrix.shape)

# Perform SVD
U, sigma, Vt = np.linalg.svd(user_item_matrix, full_matrices=False)

print(f'U shape: {U.shape}')
print(f'Sigma shape: {sigma.shape}')
print(f'Vt shape: {Vt.shape}')

In [ ]:
# Plot explained variance to choose number of latent features
explained_var_ratio = np.cumsum(sigma**2) / np.sum(sigma**2)

plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
plt.plot(range(1, len(sigma)+1), sigma, 'o-', color='steelblue', markersize=4)
plt.title('Singular Values', fontsize=13)
plt.xlabel('Latent Feature Index')
plt.ylabel('Singular Value')
plt.xlim([1, min(100, len(sigma))])
plt.grid(True, linestyle='--', alpha=0.5)

plt.subplot(1, 2, 2)
plt.plot(range(1, len(sigma)+1), explained_var_ratio, 'o-', color='darkorange', markersize=4)
plt.axhline(y=0.90, color='red', linestyle='--', label='90% variance')
plt.title('Cumulative Explained Variance', fontsize=13)
plt.xlabel('Number of Latent Features')
plt.ylabel('Cumulative Explained Variance')
plt.xlim([1, min(100, len(sigma))])
plt.legend()
plt.grid(True, linestyle='--', alpha=0.5)

plt.tight_layout()
plt.show()

n_90 = np.argmax(explained_var_ratio >= 0.90) + 1
print(f'Latent features needed for 90% variance: {n_90}')

In [ ]:
# Evaluate accuracy at different numbers of latent features
num_latent_features = [10, 50, 100, 200, 300, 500]
accuracy_scores = []

for k in num_latent_features:
    k = min(k, len(sigma))
    U_k = U[:, :k]
    sigma_k = np.diag(sigma[:k])
    Vt_k = Vt[:k, :]
    
    # Reconstruct the user-item matrix
    predicted = np.dot(np.dot(U_k, sigma_k), Vt_k)
    predicted_binary = (predicted > 0.5).astype(int)
    
    # Accuracy
    accuracy = np.mean(predicted_binary == user_item_matrix)
    accuracy_scores.append(accuracy)

plt.figure(figsize=(9, 5))
plt.plot(num_latent_features, accuracy_scores, 'gs-', markersize=8)
plt.title('SVD Reconstruction Accuracy vs. Number of Latent Features', fontsize=13)
plt.xlabel('Number of Latent Features')
plt.ylabel('Accuracy')
plt.ylim([0.9, 1.01])
plt.grid(True, linestyle='--', alpha=0.5)
plt.tight_layout()
plt.show()

for k, acc in zip(num_latent_features, accuracy_scores):
    print(f'  k={k:4d} → Accuracy: {acc:.4f}')

In [ ]:
# Choose optimal latent features
# We select k=50 as a good balance between accuracy and complexity
OPTIMAL_K = 50

U_opt = U[:, :OPTIMAL_K]
sigma_opt = np.diag(sigma[:OPTIMAL_K])
Vt_opt = Vt[:OPTIMAL_K, :]

# Item embeddings for article-article similarity
article_embeddings = Vt_opt.T  # shape: (n_articles, k)

print(f'Using {OPTIMAL_K} latent features.')
print(f'Article embeddings shape: {article_embeddings.shape}')

In [ ]:
def svd_article_recs(article_id, n=10):
    """
    Find similar articles using cosine similarity on SVD article embeddings.
    
    INPUT:
        article_id - (float/str) the source article id
        n          - (int) number of recommendations
    OUTPUT:
        rec_ids   - (list) recommended article ids
        rec_names - (list) corresponding article names
    """
    article_id = float(article_id)
    article_cols = user_item.columns.astype(float).tolist()
    
    if article_id not in article_cols:
        print(f'Article {article_id} not in user-item matrix.')
        return [], []
    
    idx = article_cols.index(article_id)
    article_vec = article_embeddings[idx].reshape(1, -1)
    
    # Cosine similarity to all articles
    sims = cosine_similarity(article_vec, article_embeddings).flatten()
    sim_series = pd.Series(sims, index=user_item.columns)
    sim_series = sim_series.drop(article_id).sort_values(ascending=False)
    
    top_ids = sim_series.head(n).index.astype(str).tolist()
    top_names = get_article_names(top_ids, df)
    return top_ids, top_names


# Test SVD recommendations
sample_id = user_item.columns[0]
sample_name = get_article_names([str(sample_id)], df)
print(f'SVD-based article-article recommendations for: {sample_name}\n')

svd_ids, svd_names = svd_article_recs(sample_id, 10)
for i, name in enumerate(svd_names, 1):
    print(f'  {i}. {name}')

---
## Discussion & Conclusions

### Summary of Methods

| Method | Description | Best For |
|--------|-------------|----------|
| **Rank-Based** | Top articles by interaction count | New users (cold-start) |
| **User-User CF** | Articles liked by similar users | Users with interaction history |
| **Content-Based** | TF-IDF + cosine similarity on text | Serendipitous discovery |
| **SVD** | Latent factor decomposition | Dense interaction matrices |

### Latent Feature Selection

We chose **k = 50** latent features because:
- The accuracy vs. k plot shows diminishing returns after ~50 features
- It captures most of the signal without overfitting to noise
- Lower k improves inference speed in production

### Cold-Start Problem

Users with no interaction history cannot use Collaborative Filtering or SVD. For these cases, we fall back to **rank-based recommendations** — the most popular articles.

### How to Test in Production

**A/B Testing** is the gold standard:
- Split users randomly into control (rank-based) and treatment (CF/SVD) groups
- Measure: click-through rate (CTR), time on page, return visit rate
- Run for at least 2 weeks to account for variance

**Offline Evaluation:**
- Hold out the last interaction of each user
- Measure whether that article appeared in the top-N recommendations (Precision@N, Recall@N, NDCG)

### Limitations

- The SVD approach assumes the user-item matrix is relatively stationary, but in reality user interests evolve
- Content-based recommendations can create a **filter bubble** — always recommending similar articles
- A **hybrid approach** (combining CF + content) would likely outperform any single method
